# Two-Tower Neural Retrieval: Managed Training and Evaluation

This notebook is the public analytical layer for the objective-conditioned two-tower retriever. Production training and evaluation logic lives in typed modules and CLI workflows. The notebook reads compact committed evidence so GitHub viewers can audit the experiment without AWS access.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "reports" / "metrics" / "two_tower_resume_proof.json").is_file()
)
resume_path = ROOT / 'reports/metrics/two_tower_resume_proof.json'
fold_path = ROOT / 'reports/metrics/two_tower_fold0_training.json'

resume = json.loads(resume_path.read_text(encoding='utf-8'))
fold = json.loads(fold_path.read_text(encoding='utf-8')) if fold_path.exists() else None
print(f"resume_proof={resume['status']} commit={resume['code_commit'][:7]}")
print(f"fold0_training={'available' if fold is not None else 'pending'}")

resume_proof=passed commit=a775bb7
fold0_training=pending


## Proven cross-worker durability

In [2]:
proof_rows = [
    ('Durable checkpoint objects', resume['checkpoint_objects']),
    ('Checkpoint bytes', f"{resume['checkpoint_bytes']:,}"),
    ('Resumed from step', resume['resumed_from_step']),
    ('Final step', resume['final_step']),
    ('Advanced steps', resume['advanced_steps']),
    ('Pipeline elapsed seconds', resume['pipeline_elapsed_seconds']),
]
pd.DataFrame(proof_rows, columns=['Invariant', 'Measured result'])

,Invariant,Measured result
0,Durable checkpoint objects,7
1,Checkpoint bytes,"2,848,858,963"
2,Resumed from step,40
3,Final step,80
4,Advanced steps,40
5,Pipeline elapsed seconds,817.041


The resume proof uses a fresh managed worker for the continuation stage. The final global step must strictly exceed the restored step; checkpoint presence alone is not counted as a pass.

## Fold 0 experimental contract

In [3]:
contract = pd.DataFrame([
    ('Held-out fold', 0),
    ('Training folds', '1, 2, 3, 4'),
    ('Candidate depths', '20 / 50 / 100 / 200 / 400 / 800'),
    ('Primary evidence', 'incremental Recall@20 ceiling vs frozen base retrieval'),
    ('Complementarity evidence', 'neural-only positive hits by objective'),
    ('Uncertainty', 'paired session-level Poisson bootstrap'),
    ('Scaling rule', 'do not run folds 1–4 unless Fold 0 adds held-out value'),
], columns=['Contract', 'Value'])
contract

,Contract,Value
0,Held-out fold,0
1,Training folds,"1, 2, 3, 4"
2,Candidate depths,20 / 50 / 100 / 200 / 400 / 800
3,Primary evidence,incremental Recall@20 ceiling vs frozen base r...
4,Complementarity evidence,neural-only positive hits by objective
5,Uncertainty,paired session-level Poisson bootstrap
6,Scaling rule,do not run folds 1–4 unless Fold 0 adds held-o...


## Fold 0 training result

In [4]:
if fold is None:
    print('Fold 0 full training has not been published yet. Run the managed fold workflow, then rerun this notebook.')
else:
    display(pd.DataFrame([
        ('status', fold['status']),
        ('global_step', fold['global_step']),
        ('completed_epochs', fold['completed_epochs']),
        ('best_valid_loss', fold['best_valid_loss']),
        ('billable_seconds', fold['billable_seconds']),
        ('checkpoint_bytes', fold['checkpoint_bytes']),
    ], columns=['Metric', 'Value']))


Fold 0 full training has not been published yet. Run the managed fold workflow, then rerun this notebook.


## Decision gate

The neural retriever advances to five-fold OOF only if its held-out candidates recover positives that the frozen revisit/co-visitation/Item2Vec system misses. Architecture complexity is not treated as evidence of quality.